# 16 — External Workloads

**Related guide:** [20_EXTERNAL_WORKLOADS.md](../docs/guides/20_EXTERNAL_WORKLOADS.md)

**Related script:** [18_external_workload.py](../examples/18_external_workload.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Wrap a foreign execution as one atomic workload and preserve its external run identity.

In [ ]:
from pyworkflowkit import TaskHandle, WorkflowRuntime, workflow
from pyworkflowkit.ecosystem import (
    ExternalWorkloadResult,
    RunContext,
    external_workload_task,
)


class ExampleRemoteJob:
    def run(self, *, context: RunContext) -> ExternalWorkloadResult:
        return ExternalWorkloadResult(
            external_run_id=f"remote-{context.task_run_id}",
            succeeded=True,
            output={"rows": 3},
        )


remote_job = external_workload_task(
    id="remote_job",
    provider="example",
    workload_ref="jobs/daily",
    workload=ExampleRemoteJob(),
)


@workflow(id="notebooks.external", version="1")
def demo() -> tuple[TaskHandle, ...]:
    return (remote_job,)

In [ ]:
runtime = WorkflowRuntime()
runtime.register(remote_job.handler_ref, remote_job.handler)
definition = demo.build()
run = runtime.run(definition)
lineage = runtime.lineage(definition, run.run_id)
external_refs = sum(len(task.external_ref_ids) for task in lineage.tasks)
print({"status": run.status.value, "external_refs": external_refs})

## Experiment

Decide explicitly whether PyWorkflowKit or the foreign system owns retry before adding retry behavior.